# Notebook 4: Search Result Evaluation
How do we know if the results of a search are on topic? 
The goal: Be able to evaluate topic search results. 

In [1]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

# Outline: 

- What is evaluation-K? Why do we set that? (search depth)
- What is precision@K
- What is recall@K? 
- What do we mean by "baseline precision" and how do you calculate it? 

## Success Metric

> **"Ratio of posts assigned to a topic that come back in topic search."**

If we run a topic-embedding search and the top results are *not* the posts that
BERTopic itself put in that topic, the topic embedding is a poor query — and the
user-facing search will surface unrelated posts.

We measure this with three numbers per topic:

1. **Precision@K** — of the top-K retrieved posts, how many were actually assigned
   to this topic? (Tracks the *display* experience — what the user sees first.)
2. **Recall@K** — of the posts assigned to this topic, how many appeared in the top-K?
   (Tracks *coverage*.)
3. **Random baseline precision** — what precision would we expect from random ranking?
   (Tells us how much our search is doing above chance.)

## Three different K values — keep them straight

(Definitions reproduced from `src/evaluation.py`.)

- **eval-K (retrieval depth)** — how many results we *retrieve* per topic for
  evaluation. This is the `size` parameter on the search query. Set it large enough
  to plausibly cover all of a topic's posts; small enough that we're not dragging in
  half the corpus. This is not included in the metrics and just used to set the
  search depth. 
- **recall-K (evaluation cutoff)** — the cutoff used in the recall calculation. Often
  equal to eval-K.
- **precision-K (display cutoff)** — typically a small number like 8 or 10 — the number
  of results a user actually sees on the first page.

Defaults in this repo: `DEFAULT_EVAL_K=100`, `DEFAULT_RECALL_K=100`, `DEFAULT_PRECISION_K=8`.

**Random baseline precision** for a topic of size *t* in a corpus of size *N* is just
`t / N` — the chance that a randomly-picked post happens to be in the topic. 

## How to compute the three metrics

Imagine a topic with 3 ground-truth posts: `{post-a, post-b, post-c}`, in a corpus of 10 posts. A
search returned 5 ranked results: `[post-a, post-b, post-c, post-d, post-e]`. 

We'll compute precision@3,
recall@3, and the random baseline by hand.

In [2]:
retrieved_ids = ["post-a", "post-b", "post-c", "post-d", "post-e"]    # search results, ranked
topic_post_ids = {"post-a", "post-c", "post-x"}             # ground truth
k = 3
dataset_size = 10

# --- Precision@K ---
# Of the top-K retrieved, how many are in the ground-truth set?
top_k = retrieved_ids[:k]                    # ['post-a', 'post-b', 'post-c']
hits = sum(pid in topic_post_ids for pid in top_k)   # post-a✓ post-b✗ post-c✓ → 2
precision_at_k = hits / len(top_k)           # 2 / 3 = 0.667
print(f"precision@{k} = {hits} hits / {len(top_k)} retrieved = {precision_at_k:.3f}")

# --- Recall@K ---
# Of the ground-truth set, how many showed up in the top-K retrieved?
recall_at_k = hits / len(topic_post_ids)     # 2 / 3 = 0.667
print(f"recall@{k}    = {hits} hits / {len(topic_post_ids)} ground-truth = {recall_at_k:.3f}")

# --- Random baseline precision ---
# If we picked posts uniformly at random, what fraction would be on-topic?
baseline = len(topic_post_ids) / dataset_size  # 3 / 10 = 0.300
print(f"baseline     = {len(topic_post_ids)} / {dataset_size} = {baseline:.3f}")
print(f"\nLift over random: {precision_at_k - baseline:+.3f}")

precision@3 = 2 hits / 3 retrieved = 0.667
recall@3    = 2 hits / 3 ground-truth = 0.667
baseline     = 3 / 10 = 0.300

Lift over random: +0.367


## Exercise: implement the three metric functions

**Time:** ~5 minutes editing + 30 seconds to verify.

Open **`src/evaluation.py`** in your editor and fill in the bodies of these three functions:

| Function | Approx. line | What it returns |
|---|---|---|
| `compute_precision_at_k` | ~59 | Fraction of top-K retrieved IDs that are in the topic |
| `compute_recall_at_k` | ~83 | Fraction of topic posts that appear in the top-K |
| `compute_random_baseline` | ~101 | `topic_size / dataset_size` |

**Edge cases — return `0.0` (don't raise, don't divide by zero):**
- Empty `retrieved_ids` → `0.0`
- Topic with zero ground-truth posts (recall denominator) → `0.0`
- Empty dataset (baseline denominator) → `0.0`

**Verify before launching the demo app — from a terminal at the repo root:**

```bash
uv run pytest -k "compute_precision_at_k or compute_recall_at_k or compute_random_baseline"
```

All matching tests should pass.

## Run the demo app

Once your tests pass, launch the Streamlit app to see the metrics in context. From a terminal at the repo root:

```bash
uv run streamlit run app.py
```

**Expected:** ~30 seconds for the first launch (longer if it preloads sample posts).
The app opens at <http://localhost:8501>.

In the app sidebar, toggle **Topic Evaluation** — for each topic you'll see
`precision@K`, `recall@K`, and the random baseline side-by-side.

**Fallback:** if you didn't finish the exercise and want to see the app working anyway,
copy the function bodies from `solutions/evaluation.py` into `src/evaluation.py`
(the app imports from `src/`, so the test-time `--solutions` swap doesn't help here).

## Interpreting the numbers

For each topic in the **Topic Evaluation** view, look for:

- Topics that beat baseline by 50× or more → search is doing real work.
- Topics that barely beat baseline → either the topic is small or the topic
  embedding is diffuse (we'll fix that in Notebook 5).
- Topics where recall@K is high but precision@K is low → the topic is well-covered
  but the *first* results aren't the best ones, often because of close-by topics.

How do different topics compare? Which topics perform well? Which don't? Why?